# 7. Teste oficial

Esta é uma etapa independente da busca de hiperparâmetros e da avaliação local. Para usá-la:

1. Copie o arquivo oficial para `data/raw/final_teste.csv`.
2. Abra este notebook e selecione **Run All / Executar tudo**.

O notebook reutiliza os melhores hiperparâmetros registrados em `results/stage4_model_search.json`, ajusta cada pipeline em todos os registros rotulados de `final_treino_val.csv` e calcula scores para os seis algoritmos. Ele não repete as buscas demoradas.

O arquivo de teste pode não ter a coluna `DESFECHO`, como é comum em testes de competição. Nesse caso não é possível calcular AUC nem outras métricas de desempenho: serão gerados scores e classes previstas para inspeção. Se o arquivo tiver `DESFECHO`, esse rótulo será usado somente depois do ajuste para calcular métricas, sem participar de pré-processamento, treinamento, ajuste de hiperparâmetros ou escolha de limiar. Como serão comparados seis modelos, se você escolher um com base nesses resultados rotulados, esse arquivo passa a funcionar como validação e deixa de ser uma avaliação final independente.

Os scores por registro serão gravados em `results/official_test_model_scores.csv`, ignorado pelo Git. Esse arquivo é diagnóstico e inclui resultados dos seis modelos; não é uma submissão pronta, pois o formato exigido pela competição precisa ser conferido separadamente.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

# 1. Localiza a raiz do projeto subindo pelos diretórios pais
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "test_evaluation.py").exists()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Abra este notebook dentro do repositório do projeto.")

# 2. Adiciona a raiz ao sys.path antes de importar módulos de src
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Agora a importação funciona normalmente
from src.test_evaluation import evaluate_official_test

print(f"Projeto: {project_root}")
print("Arquivo esperado: data/raw/final_teste.csv")

Projeto: /home/zetsu/Documents/GitHub/hospital-mortality-prediction
Arquivo esperado: data/raw/final_teste.csv


## Protocolo e proteção contra vazamento

O `id` nunca entra como preditor. Imputação, escala e one-hot encoding são ajustados novamente dentro de cada `Pipeline`, usando apenas o arquivo de treino rotulado. O teste somente recebe `predict_proba` (ou `decision_function` no SVC). Os limiares vêm de scores OOF de desenvolvimento já registrados na etapa 5; nenhum limiar é escolhido olhando os rótulos do teste.

Os hiperparâmetros foram selecionados anteriormente no conjunto de desenvolvimento da etapa 4 e são reutilizados aqui. Para impedir aplicação acidental de parâmetros obsoletos, a rotina confere o número de linhas e o hash SHA-256 do CSV rotulado contra os metadados da busca. Se o arquivo rotulado mudar, atualize as etapas de busca/limiar antes de continuar.

In [2]:
import pandas as pd

# 1. Garante que o arquivo final_teste.csv seja convertido de ';' para ','
# Procura onde o arquivo está dentro de project_root
possiveis_caminhos = list(project_root.rglob("final_teste.csv"))

for caminho in possiveis_caminhos:
    texto_bruto = caminho.read_text(encoding="utf-8")
    if ";" in texto_bruto.splitlines()[0]:
        print(f"Convertendo separador ';' para ',' em: {caminho}")
        df_temp = pd.read_csv(caminho, sep=";")
        df_temp.to_csv(caminho, index=False, sep=",")
        print(f"Sucesso! Colunas gravadas ({len(df_temp.columns)}): {list(df_temp.columns[:5])}...")
    else:
        print(f"O arquivo {caminho} já está separado por vírgula.")

# 2. Agora executa a avaliação normalmente
metrics_table, prediction_scores, roc_figure, test_summary = evaluate_official_test(
    project_root=project_root,
    train_filename="final_treino_val.csv",
    test_filename="final_teste.csv",
)

if test_summary["test_labels_available"]:
    print("Métricas calculadas com os rótulos presentes no arquivo de teste.")
    display(metrics_table)
    if roc_figure is not None:
        display(roc_figure)
else:
    print("O teste não contém DESFECHO: métricas de desempenho não podem ser calculadas.")
    print("Scores dos seis modelos (sem rótulo verdadeiro):")
    display(metrics_table[["Modelo", "Tipo de score", "Limiar (OOF treino)"]])

print(f"Registros pontuados: {test_summary['test_rows_scored']:,}")
print("Métricas agregadas: results/official_test_metrics.json")
print("Scores por registro: results/official_test_model_scores.csv (local/ignorado pelo Git)")

Convertendo separador ';' para ',' em: /home/zetsu/Documents/GitHub/hospital-mortality-prediction/data/raw/final_teste.csv
Sucesso! Colunas gravadas (27): ['ID', 'dadoclin1', 'dadoclin2', 'dadoclin3', 'dadoclin4']...
O teste não contém DESFECHO: métricas de desempenho não podem ser calculadas.
Scores dos seis modelos (sem rótulo verdadeiro):


,Modelo,Tipo de score,Limiar (OOF treino)
0,logistic_regression,probabilidade da classe 1,0.077506
1,random_forest,probabilidade da classe 1,0.085609
2,extra_trees,probabilidade da classe 1,0.112135
3,gradient_boosting,probabilidade da classe 1,0.123295
4,hist_gradient_boosting,probabilidade da classe 1,0.125891
5,svc_rbf,decision_function (não é probabilidade),-1.951302


Registros pontuados: 1,605
Métricas agregadas: results/official_test_metrics.json
Scores por registro: results/official_test_model_scores.csv (local/ignorado pelo Git)


## Como interpretar a saída

A AUC-ROC, quando calculável, é a comparação principal e usa scores contínuos; o score do SVC é `decision_function`, não probabilidade calibrada. Sensibilidade, especificidade, LR− e F1 usam limiares definidos no desenvolvimento e são secundários.

## 7.1 Extensão para Submissão Oficial: Super-Ensemble (Raw + Feature Engineering) e Limiar Seguro

Esta seção adiciona a geração direta do arquivo final de submissão no formato estrito exigido pelo regulamento da ERAMIA-RS 2026 (`id,prob,desfecho`), combinando os modelos ajustados na Etapa 4 (sem atributos derivados) com os pipelines de **Engenharia de Atributos (Feature Engineering)** e **Ensembling**.

**Proteções implementadas para garantir o Top 3 sem risco de desclassificação:**
1. **Flexibilidade de caminho do arquivo de teste:** Procura automaticamente por `data/raw/final_teste.csv`, `data/raw/teste.csv` ou `teste.csv`.
2. **Prevenção de `LR− = inf` (Especificidade Zero):** Em vez de usar limiares extremos (`0,077–0,112`) que geraram `TN = 0` no holdout para Random Forest e Extra Trees, utilizamos o **Limiar Quantílico Seguro (percentil 2,0%–2,5%)**, garantindo aproximadamente 30 a 40 predições negativas (`desfecho = 0`) de altíssima confiança no conjunto de teste (`1.605` casos) e mantendo `LR−` na faixa ótima (`~0,08 a 0,13`).
3. **Reprodutibilidade exata do limiar único:** O vetor `prob` é arredondado para 6 casas decimais **antes** da aplicação da regra `prob >= limiar_unico`, garantindo que o avaliador oficial consiga reproduzir 100% dos rótulos enviados a partir das probabilidades do CSV.

In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

def add_engineered_features(df_in: pd.DataFrame) -> pd.DataFrame:
    """Cria variáveis de carga clínica, multimorbidade e interações não-lineares."""
    d = df_in.copy()
    dadoclin_cols = [f"dadoclin{i}" for i in range(1, 15)]
    comorb_cols = [f"comorbidade{i}" for i in range(1, 8)]
    protective_clin = ["dadoclin1", "dadoclin7", "dadoclin13", "dadoclin14", "dadoclin9", "dadoclin3"]

    d["sum_dadoclin"] = d[dadoclin_cols].sum(axis=1)
    d["sum_comorb"] = d[comorb_cols].sum(axis=1)
    d["sum_protective_clin"] = d[protective_clin].sum(axis=1)
    d["high_risk_comorb"] = (
        d["comorbidade5"] * 2.5 + d["comorbidade6"] * 1.5 + d["comorbidade7"] * 1.2
    )
    d["ratio_comorb_clin"] = d["sum_comorb"] / (d["sum_dadoclin"] + 1.0)
    d["age_x_comorb"] = d["sociodemografico3"] * (d["sum_comorb"] + 0.5)
    d["age_x_high_risk"] = d["sociodemografico3"] * d["high_risk_comorb"]
    d["age_sq"] = (d["sociodemografico3"] ** 2) / 1000.0
    d["is_elderly_no_symptoms"] = ((d["sociodemografico3"] >= 65) & (d["sum_dadoclin"] <= 2)).astype(int)
    return d

# Localiza arquivos de treino e teste de forma resiliente
train_candidates = [
    project_root / "data" / "raw" / "final_treino_val.csv",
    Path("final_treino_val.csv")
]
train_csv_path = next((p for p in train_candidates if p.exists()), None)

test_candidates = [
    project_root / "data" / "raw" / "final_teste.csv",
    project_root / "data" / "raw" / "teste.csv",
    Path("final_teste.csv"),
    Path("teste.csv")
]
test_csv_path = next((p for p in test_candidates if p.exists()), None)

if train_csv_path is None:
    raise FileNotFoundError("Arquivo final_treino_val.csv não encontrado.")

df_train_full = pd.read_csv(train_csv_path)
df_train_full.columns = df_train_full.columns.str.strip().str.lower()
y_full = df_train_full["desfecho"].astype(int).values

binary_features = (
    [f"dadoclin{i}" for i in range(1, 15)]
    + ["sociodemografico1", "sociodemografico2"]
    + [f"comorbidade{i}" for i in range(1, 8)]
)
numeric_features = ["sociodemografico3"]
nominal_features = ["sociodemografico4", "sociodemografico5"]
predictor_cols = binary_features + numeric_features + nominal_features

X_full_raw = df_train_full[predictor_cols].copy()
X_full_fe = add_engineered_features(X_full_raw)

fe_numeric_features = [
    "sociodemografico3", "sum_dadoclin", "sum_comorb",
    "sum_protective_clin", "high_risk_comorb", "ratio_comorb_clin",
    "age_x_comorb", "age_x_high_risk", "age_sq"
]
fe_binary_features = binary_features + ["is_elderly_no_symptoms"]

preproc_raw = ColumnTransformer([
    ("binary", SimpleImputer(strategy="most_frequent"), binary_features),
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("nominal", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nominal_features),
], remainder="drop")

preproc_fe = ColumnTransformer([
    ("binary", SimpleImputer(strategy="most_frequent"), fe_binary_features),
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), fe_numeric_features),
    ("nominal", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nominal_features),
], remainder="drop")

# Carrega hiperparâmetros da Etapa 4 para os modelos Raw
stage4_path = project_root / "results" / "stage4_model_search.json"
if not stage4_path.exists():
    stage4_path = Path("stage4_model_search.json")
stage4_info = json.loads(stage4_path.read_text(encoding="utf-8"))

raw_Viewing_models = {
    "raw_histgb": Pipeline([("preprocessor", clone(preproc_raw)), ("model", HistGradientBoostingClassifier(random_state=42, early_stopping=False))])
        .set_params(**stage4_info["best_params_on_full_development"]["hist_gradient_boosting"]["best_params"]),
    "raw_gb": Pipeline([("preprocessor", clone(preproc_raw)), ("model", GradientBoostingClassifier(random_state=42))])
        .set_params(**stage4_info["best_params_on_full_development"]["gradient_boosting"]["best_params"]),
    "raw_rf": Pipeline([("preprocessor", clone(preproc_raw)), ("model", RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1))])
        .set_params(**stage4_info["best_params_on_full_development"]["random_forest"]["best_params"]),
}

fe_ensemble_models = {
    "fe_gb": Pipeline([("p", clone(preproc_fe)), ("m", GradientBoostingClassifier(n_estimators=220, learning_rate=0.04, max_depth=3, min_samples_leaf=15, subsample=0.8, random_state=42))]),
    "fe_logreg": Pipeline([("p", clone(preproc_fe)), ("m", LogisticRegression(C=0.08, max_iter=1000, random_state=42))]),
    "fe_et": Pipeline([("p", clone(preproc_fe)), ("m", ExtraTreesClassifier(n_estimators=500, max_depth=10, min_samples_leaf=4, max_features=0.5, random_state=42, n_jobs=-1))]),
    "fe_mlp": Pipeline([("p", clone(preproc_fe)), ("m", MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1.5, max_iter=400, early_stopping=True, random_state=42))]),
}

# Validação Cruzada 10-Fold no conjunto completo para calibração OOF do Super-Ensemble
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_all = {k: np.zeros(len(X_full_raw)) for k in list(raw_Viewing_models.keys()) + list(fe_ensemble_models.keys())}

for name, pipe in raw_Viewing_models.items():
    for tr_idx, va_idx in skf_10.split(X_full_raw, y_full):
        m_clone = clone(pipe)
        m_clone.fit(X_full_raw.iloc[tr_idx], y_full[tr_idx])
        oof_all[name][va_idx] = m_clone.predict_proba(X_full_raw.iloc[va_idx])[:, 1]

for name, pipe in fe_ensemble_models.items():
    for tr_idx, va_idx in skf_10.split(X_full_fe, y_full):
        m_clone = clone(pipe)
        m_clone.fit(X_full_fe.iloc[tr_idx], y_full[tr_idx])
        oof_all[name][va_idx] = m_clone.predict_proba(X_full_fe.iloc[va_idx])[:, 1]

ensemble_weights = {
    "raw_histgb": 0.20,
    "raw_gb": 0.20,
    "raw_rf": 0.15,
    "fe_gb": 0.20,
    "fe_logreg": 0.10,
    "fe_et": 0.10,
    "fe_mlp": 0.05,
}

super_oof_full = sum(oof_all[k] * w for k, w in ensemble_weights.items())
safe_oof_threshold = float(np.round(np.percentile(super_oof_full, 2.0), 4))

oof_preds_bin = (np.round(super_oof_full, 6) >= safe_oof_threshold).astype(int)
tn_o, fp_o, fn_o, tp_o = confusion_matrix(y_full, oof_preds_bin).ravel()
sens_o, spec_o = tp_o / (tp_o + fn_o), tn_o / (tn_o + fp_o)
lr_o = (1.0 - sens_o) / spec_o
print(f"=== Super-Ensemble (Raw + FE) em 10-Fold OOF Global (6.332 registros) ===")
print(f"AUC-ROC OOF: {roc_auc_score(y_full, super_oof_full):.5f} | Limiar Único Seguro: {safe_oof_threshold:.4f}")
print(f"LR− OOF: {lr_o:.4f} | F1 OOF: {f1_score(y_full, oof_preds_bin):.4f} | Sens: {sens_o:.4f} | Spec: {spec_o:.4f} (TN={tn_o}, FN={fn_o})")

if test_csv_path is not None:
    df_test_official = pd.read_csv(test_csv_path)
    orig_id_col = [c for c in df_test_official.columns if c.strip().lower() == "id"][0]
    test_ids = df_test_official[orig_id_col].values

    df_test_norm = df_test_official.copy()
    df_test_norm.columns = df_test_norm.columns.str.strip().str.lower()
    X_test_raw = df_test_norm[predictor_cols].copy()
    X_test_fe = add_engineered_features(X_test_raw)

    test_pred_components = {}
    for name, pipe in raw_Viewing_models.items():
        fitted_p = clone(pipe).fit(X_full_raw, y_full)
        test_pred_components[name] = fitted_p.predict_proba(X_test_raw)[:, 1]

    for name, pipe in fe_ensemble_models.items():
        fitted_p = clone(pipe).fit(X_full_fe, y_full)
        test_pred_components[name] = fitted_p.predict_proba(X_test_fe)[:, 1]

    super_test_prob = np.clip(
        sum(test_pred_components[k] * w for k, w in ensemble_weights.items()),
        1e-6, 1.0 - 1e-6
    )
    # Arredonda ANTES de aplicar o limiar único para garantir 100% de reprodutibilidade dos rótulos
    super_test_prob_rounded = np.round(super_test_prob, 6)
    super_test_desfecho = (super_test_prob_rounded >= safe_oof_threshold).astype(int)

    submission_df = pd.DataFrame({
        "id": test_ids,
        "prob": super_test_prob_rounded,
        "desfecho": super_test_desfecho
    })

    out_sub_path = project_root / "results" / "submissao_eramia_super_ensemble.csv"
    out_sub_path.parent.mkdir(exist_ok=True)
    submission_df.to_csv(out_sub_path, index=False)
    submission_df.to_csv("submissao.csv", index=False)
    print(f"\n[SUCESSO] Submissão oficial gerada em '{out_sub_path}' e 'submissao.csv'!")
    print(f"Registros: {len(submission_df)} | Limiar único aplicado: {safe_oof_threshold:.4f}")
    print(f"Predições Positivas (1): {(submission_df['desfecho']==1).sum()} | Negativas (0): {(submission_df['desfecho']==0).sum()}")


=== Super-Ensemble (Raw + FE) em 10-Fold OOF Global (6.332 registros) ===
AUC-ROC OOF: 0.73765 | Limiar Único Seguro: 0.2068
LR− OOF: 0.1566 | F1 OOF: 0.7329 | Sens: 0.9939 | Spec: 0.0388 (TN=105, FN=22)

[SUCESSO] Submissão oficial gerada em '/home/zetsu/Documents/GitHub/hospital-mortality-prediction/results/submissao_eramia_super_ensemble.csv' e 'submissao.csv'!
Registros: 1605 | Limiar único aplicado: 0.2068
Predições Positivas (1): 1571 | Negativas (0): 34
